In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import cpca
import os
import math
os.chdir("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/抗议数据")


In [ ]:
df = pd.read_csv("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/抗议数据/RFA_protest3.csv",encoding='utf-8')
df = df[['adcode','location','size_level','year','month','day','citycode','省','市','区']]
df.to_csv("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/抗议数据/RFA_protest3_cropped.csv",index=False)

In [6]:
df = pd.read_stata("cloudseeding.dta")
df["district"] = df['prov'] + df['city'] + df['county']
df_adcode = cpca.transform(df["district"])
df['adcode'] = df_adcode['adcode']
df.to_csv("cloudseeding_adcode.csv")

In [12]:
df = pd.read_stata("Meteorological.dta")
df["district"] = df['prov'] + df['city'] + df['county']
df_adcode = cpca.transform(df["district"])
df['adcode'] = df_adcode['adcode']
df.to_csv("meteorological_adcode.csv")

In [24]:
df = pd.read_stata("final_panel_newweibo.dta")
df = df.sort_values(by=['adcode', 'date']).reset_index()
df["day"] = df.groupby("adcode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for ad, ad_df in df.groupby('adcode'):
    protest_dates = ad_df.loc[ad_df['n_prt_weibo'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['adcode'] == int(ad)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_weibo_county.csv')

In [25]:
df = pd.read_stata("final_panel_newweibo.dta")
df = df.sort_values(by=['adcode', 'date']).reset_index()
df["day"] = df.groupby("adcode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for ad, ad_df in df.groupby('adcode'):
    protest_dates = ad_df.loc[ad_df['n_prt_rfa'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['adcode'] == int(ad)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_rfa_county.csv')

In [ ]:
df = pd.read_stata("eventstudy_city.dta")
df = df.sort_values(by=['citycode', 'date']).reset_index()
df["day"] = df.groupby("citycode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for city, city_df in df.groupby('citycode'):
    protest_dates = city_df.loc[city_df['n_prt_weibo'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['citycode'] == int(city)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_weibo_city.csv')

In [12]:
df = pd.read_stata("eventstudy_city.dta")
df = df.sort_values(by=['citycode', 'date']).reset_index()
df["day"] = df.groupby("citycode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for city, city_df in df.groupby('citycode'):
    protest_dates = city_df.loc[city_df['n_prt_rfa'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['citycode'] == int(city)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_rfa_city.csv')

In [ ]:
df = pd.read_csv('eventstudy_weibo_city.csv')

# Identify unique events and assign an event ID 
df_list = []
for city, city_df in df.groupby('citycode'):
    city_df['event_id'] = (city_df['event'] == 1).cumsum()
    city_df.loc[df['event'] == 0, 'event_id'] = None   
    num = city_df['event_id'].max()

    if math.isnan(num):
        city_df['num'] = 0
        df_list.append(city_df)
    else:
        for i in range(0,int(num)):
            city_df_temp = city_df.copy()
            city_df_temp['num'] = i+1
            df_list.append(city_df_temp)

df_final = pd.concat(df_list)
df_final.to_stata('temp_event.dta')

In [13]:
df = pd.read_csv('eventstudy_rfa_city.csv')

# Identify unique events and assign an event ID 
df_list = []
for city, city_df in df.groupby('citycode'):
    city_df['event_id'] = (city_df['event'] == 1).cumsum()
    city_df.loc[df['event'] == 0, 'event_id'] = None   
    num = city_df['event_id'].max()

    if math.isnan(num):
        city_df['num'] = 0
        df_list.append(city_df)
    else:
        for i in range(0,int(num)):
            city_df_temp = city_df.copy()
            city_df_temp['num'] = i+1
            df_list.append(city_df_temp)

df_final = pd.concat(df_list)
df_final.to_stata('temp_event_rfa.dta')


/var/folders/5h/rb902v091ys_0c5x0jt8sbjw0000gp/T/ipykernel_71010/1475265588.py:20: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    Unnamed: 0   ->   Unnamed__0

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_final.to_stata('temp_event_rfa.dta')


In [ ]:
import pandas as pd
import os
import math
os.chdir("/Users/anora/Library/CloudStorage/Dropbox-TeamMG/Wanru Wu/Cloudseeding_Anora/抗议数据")


df = pd.read_stata("eventstudy_city.dta")
df = df.sort_values(by=['citycode', 'date']).reset_index()
df["day"] = df.groupby("citycode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for city, city_df in df.groupby('citycode'):
    protest_dates = city_df.loc[city_df['n_prt_rfa'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['citycode'] == int(city)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_rfa_city.csv')

df = pd.read_csv('eventstudy_rfa_city.csv')

# Identify unique events and assign an event ID 
df_list = []
for city, city_df in df.groupby('citycode'):
    city_df['event_id'] = (city_df['event'] == 1).cumsum()
    city_df.loc[df['event'] == 0, 'event_id'] = None   
    num = city_df['event_id'].max()

    if math.isnan(num):
        city_df['num'] = 0
        df_list.append(city_df)
    else:
        for i in range(0,int(num)):
            city_df_temp = city_df.copy()
            city_df_temp['num'] = i+1
            df_list.append(city_df_temp)

df_final = pd.concat(df_list)
df_final.to_stata('temp_event_rfa.dta')

/var/folders/5h/rb902v091ys_0c5x0jt8sbjw0000gp/T/ipykernel_2159/1093204778.py:50: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    Unnamed: 0   ->   Unnamed__0

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  df_final.to_stata('temp_event_rfa.dta')


In [2]:
df = pd.read_stata("eventstudy_city.dta")
df = df.sort_values(by=['citycode', 'date']).reset_index()
df["day"] = df.groupby("citycode").cumcount()
df['event'] = 0

# Identify events (first protest and subsequent protests >= 3 months apart)
for city, city_df in df.groupby('citycode'):
    protest_dates = city_df.loc[city_df['n_prt_weibo'] > 0, 'day'].sort_values().tolist()
    last_event = None
    
    for protest_date in protest_dates:
        if last_event is None or (protest_date - last_event) >= 45:
            df.loc[(df['citycode'] == int(city)) & (df['day'] == protest_date),'event'] = 1
            last_event = protest_date

index_list = df.loc[df['event']==1,'index'].tolist()
df['to_day']=None
for index in index_list:
    for i in range(-22,23):
        if not (index+i<0) or (index+i>len(df)):
            df.loc[df['index']==index+i,'to_day'] = i

df.to_csv('eventstudy_weibo_city.csv')